# S17 · CVaR: how bad is a bad day, on average?

In notebook 1 we found the **95% VaR** of our ₹10 lakh portfolio: the loss that only 1
day in 20 goes beyond. But VaR goes silent past its line. It does not say *how much*
worse those days are.

**CVaR** (also called Expected Shortfall) fills the gap. It is the average loss on the
days past the VaR line. If VaR is the edge of the cliff, CVaR is how far you fall.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each
  cell, top to bottom, and read the plain-English note above each one.
- New to "quantile" or "the tail"? Open the primer `primers/quantiles_and_tails.md`
  for a ten-minute, picture-first version.
- Already confident with code or with probability? Look for the cells marked
  **Stretch (optional)** near the end.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook downloads stock prices with yfinance, which Colab does not
# always ship. numpy, pandas and matplotlib are already there.
import sys
if "google.colab" in sys.modules:
    !pip install -q yfinance
else:
    print("Not on Colab - assuming the libraries are already installed.")

## Step 1 — get our tools ready

NumPy does maths on long lists of numbers, pandas holds tables of data, and
matplotlib draws charts.

In [ ]:
from pathlib import Path
import numpy as np                 # fast maths on lists of numbers
import pandas as pd                # tables of data
import matplotlib.pyplot as plt    # charts

print("Tools ready.")

## Step 2 — download the prices

Our portfolio is the basket from S16: five well-known NSE stocks. We download three
years of daily closing prices, from January 2023 to December 2025. The dates are
fixed, so you get the same numbers as the slides.

If there is no internet (say, in an exam hall), the cell loads a saved copy of exactly
the same prices instead, so the notebook always runs. It prints which one happened.

In [ ]:
# The five stocks. The '.NS' ending means the National Stock Exchange of India.
tickers = ["TCS.NS", "INFY.NS", "RELIANCE.NS", "ITC.NS", "HDFCBANK.NS"]

# Where the saved copy lives: next to this notebook, or on the course website.
local_copy = Path("../data/nse_five_stocks_2023_2025.csv")
website_copy = ("https://girishmkulkarni.github.io/applied-maths-in-industry-site/"
                "sessions/S17/data/nse_five_stocks_2023_2025.csv")

try:
    import yfinance as yf

    # Daily prices for all five stocks at once. We keep the closing price.
    prices = yf.download(tickers, start="2023-01-01", end="2026-01-01",
                         auto_adjust=True, progress=False)["Close"]

    # Keep our column order and drop any day with a missing price.
    prices = prices[tickers].dropna()

    # An empty table means the download quietly failed.
    if len(prices) < 100:
        raise ValueError("the download came back empty")
    print("Downloaded live prices from Yahoo Finance.")

except Exception as problem:
    print("Could not download (", problem, ") - loading the saved copy instead.")
    if local_copy.exists():
        prices = pd.read_csv(local_copy, index_col=0, parse_dates=True)
    else:
        prices = pd.read_csv(website_copy, index_col=0, parse_dates=True)

print("price table:", prices.shape[0], "days,", prices.shape[1], "stocks")
prices.head()

## Step 3 — turn prices into rupee losses

Three small moves:

1. A **return** is the percentage change in price from one day to the next.
   `pct_change()` works it out for every stock and every day.
2. We put ₹2 lakh in each stock, so each has a **weight** of 0.2 (one fifth of the
   money). `returns @ weights` mixes the five returns into one portfolio return per day,
   the same `@` you met in S16.
3. Risk is about money we *lose*, so we flip the sign and multiply by the ₹10 lakh
   invested. A 1% fall becomes a loss of ₹10,000. From now on, a bigger number is worse.

In [ ]:
# Daily return of each stock: (today's price - yesterday's) / yesterday's.
returns = prices.pct_change().dropna()

# Rs 2 lakh in each of the five stocks, out of Rs 10 lakh.
weights = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
investment = 1_000_000

# One portfolio return per day, flipped into a rupee LOSS.
# A negative return (a fall) becomes a positive loss.
losses = -(returns @ weights) * investment

print("days of data:", len(losses))
print("worst day   :", round(losses.max()), "rupees lost on", losses.idxmax().date())
print("best day    :", round(-losses.min()), "rupees gained on", losses.idxmin().date())

## Step 4 — the 95% VaR, as before

Same one line as notebook 1.

In [ ]:
var_hist = np.percentile(losses, 95)

print("Historical 95% VaR:", round(var_hist), "rupees")

## Step 5 — VaR's blind spot

Keep only the days at or past the VaR line: the **tail**. VaR treats all of them as
one fact, "past the line". But look how different they are.

In [ ]:
# The tail: every day whose loss is at least the VaR.
tail = losses[losses >= var_hist]

print("days in the tail:", len(tail))
print("smallest tail loss:", round(tail.min()), "rupees")
print("biggest tail loss :", round(tail.max()), "rupees")
print("the worst day crossed the VaR line by", round(tail.max() - var_hist), "rupees")

## Step 6 — CVaR: average the tail

CVaR is just the average of those tail losses. We do it twice: once by hand (add them
up and divide by how many there are) and once with `.mean()`. Same number.

In [ ]:
# By hand: total of the tail losses, divided by how many there are.
cvar_by_hand = tail.sum() / len(tail)

# The one-liner.
cvar_hist = tail.mean()

print("CVaR by hand   :", round(cvar_by_hand), "rupees")
print("CVaR with mean :", round(cvar_hist), "rupees")
print("\n95% VaR :", round(var_hist), "rupees  (where the bad days start)")
print("95% CVaR:", round(cvar_hist), "rupees  (what a bad day costs on average)")

CVaR is always at least as big as VaR, because every number in the average is at least
the VaR.

## Step 7 — draw VaR and CVaR together

The dashed red line is VaR, where the bad zone starts. The solid purple line is CVaR,
the average of everything in the red tail. CVaR always sits to the right.

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(losses / 1000, bins=40, color="#2E75B6", alpha=0.6)
plt.hist(tail / 1000, bins=15, color="#C0392B")

plt.axvline(var_hist / 1000, color="#C0392B", linestyle="--", linewidth=2.5,
            label="95% VaR: where the bad days start")
plt.axvline(cvar_hist / 1000, color="#6A4C93", linewidth=2.5,
            label="95% CVaR: the average bad day")

plt.xlabel("loss that day, thousand rupees")
plt.ylabel("number of days")
plt.title("VaR is the edge of the cliff; CVaR is how far you fall")
plt.legend()
plt.show()

## Step 8 — same VaR, different CVaR

Here is why CVaR matters. Imagine putting the whole ₹10 lakh into **one** stock
instead. We compare ITC with HDFC Bank, using the same two lines for each.

In [ ]:
for stock in ["ITC.NS", "HDFCBANK.NS"]:
    # Losses if all Rs 10 lakh sat in this one stock.
    stock_losses = -returns[stock] * investment

    stock_var = np.percentile(stock_losses, 95)
    stock_cvar = stock_losses[stock_losses >= stock_var].mean()

    print(stock, "  VaR:", round(stock_var), "  CVaR:", round(stock_cvar))

The two VaRs are within a few hundred rupees of each other. HDFC Bank's is even a little
lower, so a report that quotes only VaR would call it the safer stock. But its CVaR is
thousands of rupees higher: its bad days go much deeper. The picture shows why.

In [ ]:
plt.figure(figsize=(9, 5))

for stock, colour in [("ITC.NS", "#2E7D32"), ("HDFCBANK.NS", "#C0392B")]:
    stock_losses = -returns[stock] * investment
    stock_var = np.percentile(stock_losses, 95)

    # This stock's tail days, lined up worst first.
    worst_days = stock_losses[stock_losses >= stock_var].sort_values(ascending=False)
    plt.plot(range(1, len(worst_days) + 1), worst_days.values / 1000, "-o",
             color=colour, markersize=4, label=stock)

plt.xlabel("the worst days for each stock, worst first")
plt.ylabel("loss that day, thousand rupees")
plt.title("Same VaR, very different bad days")
plt.legend()
plt.show()

Both lines end at about the same place (the VaR), but HDFC Bank's worst days start
far higher. On 17 January 2024 it fell about 8.4% after its results. VaR cannot see
that. CVaR can.

### Stretch (optional) — VaR and CVaR at three confidence levels

Skip this if you are new to code. A risk desk rarely looks at just one level. This loop
computes both numbers at 90%, 95% and 99% for our portfolio. Both climb as we go deeper
into the tail, and CVaR stays thousands of rupees above VaR at every level.

In [ ]:
for level in [90, 95, 99]:
    level_var = np.percentile(losses, level)
    level_cvar = losses[losses >= level_var].mean()
    print(level, "%   VaR:", round(level_var), "  CVaR:", round(level_cvar),
          "  gap:", round(level_cvar - level_var))

## Your turn

Which of our five stocks has the biggest gap between its CVaR and its VaR? Adapt the
loop in Step 8 to run over all five `tickers` and print the gap for each. What does a
big gap tell a risk manager?

In [ ]:
# Your code here: loop over all five tickers and print CVaR - VaR for each.

## What you just did

You found VaR's blind spot, fixed it with **CVaR** (the average loss past the VaR line),
and saw on real data that two stocks with the same VaR can hide very different bad days.
In practice a desk reports both: VaR for where the bad days start, CVaR for how deep
they go.

Notebook 3 builds the risk engine banks actually run: Monte Carlo simulation, plus a
stress test and a backtest.